# Magic: Measuring Non-Stabilizerness

Some quantum computations are easy to simulate on a classical computer and some are not. This tutorial introduces the quantity that draws that line<sup><a href="#note-entanglement">†</a></sup>, called **magic** or **non-stabilizerness**. The following sections walk through the concepts formally and with hands-on numerical examples using the quax package.

No prior knowledge of magic is assumed. You should be comfortable with state vectors, density matrices and the Pauli matrices.

## The Stabilizer Formalism

For a qubit, the $X$ Pauli matrix flips its bit and the $Z$ matrix flips its phase. A qudit of dimension $d$ generalizes these transitions:

$$X|j\rangle = |j + 1 \bmod d\rangle, \qquad Z|j\rangle = \omega^j |j\rangle, \qquad \omega = e^{2\pi i/d}.$$

The **Weyl operators** are their products with a fixed phase, for $x, z \in \{0, \dots, d-1\}$:

$$W_{x,z} = \tau^{xz} X^x Z^z, \qquad \tau = (-1)^d e^{i\pi/d}.$$

For a qubit $\tau = i$, so $W_{1,1} = iXZ = Y$ and the four Weyl operators are the Paulis $I, X, Y, Z$. For odd $d$ they are Gross's Weyl operators<sup><a href="#ref-1">1</a></sup>. This convention, the one quax's characteristic function and magic measures use, is Eq. (2.1) of Gross, Nezami and Walter<sup><a href="#ref-2">2</a></sup>. Other conventions exist, differing in the phase and in the order of $X$ and $Z$<sup><a href="#ref-3">3</a></sup>. On several qudits the Weyl operators are tensor products, $D^2$ of them for total dimension $D$. They are orthogonal, so any operator is a unique combination of them.

**Stabilizer states.** A pure stabilizer state is the common eigenstate, with eigenvalue 1, of a group of $D$ commuting Weyl operators (up to phases). For a qubit, $|0\rangle$ is stabilized by $Z$ and $|+\rangle$ by $X$; the Bell state $(|00\rangle + |11\rangle)/\sqrt{2}$ is stabilized by $X \otimes X$ and $Z \otimes Z$. A list of generators of the group describes the state completely<sup><a href="#ref-4">4</a></sup>. Mixtures of pure stabilizer states are stabilizer states too; the set of all of them, pure and mixed, is written STAB<sup><a href="#ref-3">3</a></sup>. For one qubit, STAB is the octahedron in the Bloch ball whose corners are the six pure stabilizer states.

**Clifford gates and stabilizer operations.** A Clifford gate maps every Weyl operator, under conjugation, to another Weyl operator up to a phase, so it maps stabilizer states to stabilizer states. Hadamard, phase and CNOT are Clifford gates for qubits. The **stabilizer operations** are Clifford gates, adding stabilizer states, measuring in the computational basis, discarding qudits, and classical control on measurement outcomes (Definition 2 of Veitch et al.<sup><a href="#ref-3">3</a></sup>).

By the Gottesman–Knill theorem, stabilizer operations can be simulated efficiently on a classical computer, by updating the short list of generators instead of the full state vector<sup><a href="#ref-4">4</a></sup>.

## Magic as a Resource

Stabilizer operations alone are classically simulable, so they cannot be all a quantum computer does in order to achieve quantum advantage. Adding a suitable non-Clifford gate, such as the T gate, to the Clifford gates makes the gate set universal<sup><a href="#ref-4">4</a></sup>. The property that non-Clifford gates and non-stabilizer states have, and stabilizer operations lack, is called **magic**. Only circuits that need magic can be hard to simulate on a classical computer<sup><a href="#note-entanglement">†</a></sup>, and measuring magic tells us how far a state is from the classically easy stabilizer states.

Veitch et al. define it by exclusion (Definition 1 of ref.<sup><a href="#ref-3">3</a></sup>):

> A state is magic if it is not a stabilizer state.

Because STAB contains mixtures, noise can push a magic state into STAB, and then its magic is gone.

The standard example, used throughout this notebook, is the T state:

$$|T\rangle = T|+\rangle = \frac{|0\rangle + e^{i\pi/4}|1\rangle}{\sqrt{2}}.$$

Bravyi and Kitaev, who coined the term *magic state*, use *T-type* for a different state, along $(1, 1, 1)/\sqrt{3}$ in the Bloch sphere; today's T state is a Clifford rotation of their *H-type* state<sup><a href="#ref-5">5</a></sup>.

This suggests treating magic as a **resource**, similar to how entanglement is treated. Stabilizer operations are free, and magic is what they cannot create. A **magic monotone** seeks to measure this precisely. It is a number assigned to each state that stabilizer operations cannot increase on average<sup><a href="#ref-3">3</a></sup>. The rest of the notebook introduces several monotones and related functions.

<small><a id="note-entanglement"></a>† Magic is necessary for a computation to be hard to simulate classically, but not sufficient: it also needs entanglement. Stabilizer circuits can be highly entangled and still easy to simulate, and qubits that never become entangled can be simulated one at a time.</small>

## Setup

Turn on 64-bit precision before computing anything.

In [ ]:
import jax
import jax.numpy as jnp

# Before importing quax, so its predefined gates and states are 64-bit too
jax.config.update("jax_enable_x64", True)

import quax as qx

# Print arrays with three decimals and without tiny rounding noise
jnp.set_printoptions(precision=3, suppress=True)

## Weyl Operators and the Characteristic Function

quax orders the $d^2$ Weyl operators of a qudit in a fixed way, and `qx.weyl_basis_labels` names them `W{x}{z}` in that order. For a qubit, `qx.weyl_basis(2)` returns the operators themselves: $I, X, Y, Z$.

In [ ]:
for label, operator in zip(qx.weyl_basis_labels(2), qx.weyl_basis(2).matrix):
    print(label)
    print(operator)
    print()

For a qutrit, `qx.weyl_basis(3)` returns the plain products $X^x Z^z$, and `qx.weyl_phases` gives the phase $\tau^{xz}$ that turns each one into $W_{x,z}$.

In [ ]:
labels = qx.weyl_basis_labels(3)
phases = qx.weyl_phases((3,))

# W_{x,z} = tau^{xz} X^x Z^z
weyl_operators = phases[:, None, None] * qx.weyl_basis(3).matrix

for label, phase, operator in zip(labels, phases, weyl_operators):
    print(label, "phase", phase)
    print(operator)

The **characteristic function** of a state lists its expectation value for every Weyl operator, in the same order:

$$\chi_\rho(x, z) = \mathrm{Tr}[W_{x,z}\,\rho].$$

In [ ]:
# Zero state
print("|0>", qx.characteristic_function(qx.states.KET0))

# Plus state
print("|+>", qx.characteristic_function(qx.states.KETPLUS))

# T state, T|+>
t_state = qx.apply_unitary_to_state_vector(qx.gates.T, qx.states.KETPLUS)
print("|T>", qx.characteristic_function(t_state))

$|0\rangle$ has $\langle Z \rangle = 1$ and $|+\rangle$ has $\langle X \rangle = 1$: each has two entries of modulus 1, the identity and its stabilizer, and zeros elsewhere. The T state spreads its weight over $X$ and $Y$, with $1/\sqrt{2}$ on each.

Squaring the moduli and dividing by $D$ gives the **characteristic distribution**

$$\Xi_\rho(x, z) = \frac{|\chi_\rho(x, z)|^2}{D}.$$

It sums to the purity $\mathrm{Tr}[\rho^2]$, so for a pure state it is a probability distribution over the Weyl operators. A stabilizer state spreads it evenly over its $D$ stabilizers. A magic state spreads it wider. The stabilizer norm and the stabilizer Rényi entropies below measure that spread, for any $d$.

In [ ]:
# Zero state
print("|0>", qx.characteristic_distribution(qx.states.KET0))

# T state
print("|T>", qx.characteristic_distribution(t_state))

## Stabilizer States

`qx.stabilizer_states(dims)` lists every pure stabilizer state of qudits with prime dimensions, as one `StateVector` ensemble. It applies Clifford gates to $|0\cdots0\rangle$ until no new state appears, and checks the result against Gross's count $d^n \prod_{i=1}^{n} (d^i + 1)$ for $n$ qudits of dimension $d$ (Corollary 21 of ref.<sup><a href="#ref-1">1</a></sup>).

For one qubit there are six: the eigenstates of $Z$, $X$ and $Y$.

In [ ]:
# The six single-qubit stabilizer states
six = qx.stabilizer_states((2,))
print(six.matrix)

In [ ]:
# How many stabilizer states there are
print("one qubit:   ", qx.stabilizer_states((2,)).matrix.shape[0])
print("one qutrit:  ", qx.stabilizer_states((3,)).matrix.shape[0])
print("two qubits:  ", qx.stabilizer_states((2, 2)).matrix.shape[0])
print("two qutrits: ", qx.stabilizer_states((3, 3)).matrix.shape[0])

Every one of them has $|\chi| = 1$ on $D$ Weyl operators and $0$ on the rest. They are the zero-magic reference: each measure below gives them its zero-magic value.

In [ ]:
# |chi| for each of the six states, one row per state
print(jnp.abs(qx.characteristic_function(six)))

## Magic Depends on the Frame

Everything so far rests on one choice: which states are named $|0\rangle, \dots, |d-1\rangle$. Hardware makes that choice through the basis it measures in. $X$, $Z$, the Weyl operators, the Clifford group and the stabilizer states all follow from it.

Rotate every qudit by a unitary $V$ and you get a second stabilizer formalism, just as valid, whose stabilizer states are $V|s\rangle$. To measure magic in that frame, rotate back first:

$$M_V(\rho) = M(V^\dagger \rho\, V).$$

For example, $V = TH$ sends $|0\rangle$ to the T state, so in that frame the T state is a stabilizer state:

In [ ]:
# The frame rotation V = TH
V = qx.gates.T @ qx.gates.H

# Rotate the T state back by V-dagger (V.T is the transpose, not the T gate)
rotated_back = qx.apply_unitary_to_state_vector(V.T.conj(), t_state)

print(jnp.abs(qx.characteristic_function(rotated_back)))

Its characteristic function now matches $|0\rangle$'s. The rotated theory has the same shape as the original: its Clifford circuits are just as easy to simulate, and every measure gives $V|\psi\rangle$ the value $|\psi\rangle$ had. Only the names change. On several qudits, $V$ should be a tensor product of single-qudit rotations; an entangling $V$ would also change what counts as a separate qudit.

## The Stabilizer Norm

The simplest Weyl-based measure averages the moduli of the characteristic function:

$$\mathcal{D}(\rho) = \frac{1}{D} \sum_{x,z} |\mathrm{Tr}[W_{x,z}\,\rho]|.$$

Campbell introduced it for qubits (Lemma 1 of ref.<sup><a href="#ref-6">6</a></sup>), and Howard and Campbell used it to bound the cost of magic-state protocols<sup><a href="#ref-7">7</a></sup>. quax applies the same formula to qudits.

How to read it:

- **Pure states.** Every $|\chi| \le 1$ and $\sum |\chi|^2 = D$, so $\mathcal{D} \ge 1$, and $\mathcal{D} = 1$ exactly when the state is a stabilizer state.
- **Mixed states.** Every state in STAB has $\mathcal{D} \le 1$, so $\mathcal{D} > 1$ proves a state is magic. $\mathcal{D} \le 1$ proves nothing.

In [ ]:
# Zero state
print("|0>", qx.stabilizer_norm(qx.states.KET0))

# Plus state
print("|+>", qx.stabilizer_norm(qx.states.KETPLUS))

# T state
print("|T>", qx.stabilizer_norm(t_state))

# Maximally mixed state
maximally_mixed = qx.DensityMatrix.from_matrix(jnp.eye(2, dtype=complex) / 2, (2,))
print("I/2", qx.stabilizer_norm(maximally_mixed))

The T state gets $(1 + \sqrt{2})/2 \approx 1.207$. Mixing it with noise, $\rho_p = (1 - p)\,|T\rangle\langle T| + p\,I/2$, shrinks every non-identity entry of $\chi$ by $1 - p$:

In [ ]:
t_matrix = qx.promote_state_vector_to_density_matrix(t_state).matrix
mixed_matrix = jnp.eye(2) / 2

# 20% noise
noisy_t = qx.DensityMatrix.from_matrix(0.8 * t_matrix + 0.2 * mixed_matrix, (2,))
print("p = 0.2", qx.stabilizer_norm(noisy_t))

# 30% noise
noisy_t = qx.DensityMatrix.from_matrix(0.7 * t_matrix + 0.3 * mixed_matrix, (2,))
print("p = 0.3", qx.stabilizer_norm(noisy_t))

# 50% noise
noisy_t = qx.DensityMatrix.from_matrix(0.5 * t_matrix + 0.5 * mixed_matrix, (2,))
print("p = 0.5", qx.stabilizer_norm(noisy_t))

The norm crosses 1 at $p = 1 - 1/\sqrt{2} \approx 0.29$. For a single qubit that is exactly where the noisy T state enters STAB: with Bloch vector $(x, y, z)$, $\mathcal{D} = (1 + |x| + |y| + |z|)/2$, and STAB is the octahedron $|x| + |y| + |z| \le 1$.

With more qubits the norm can miss magic. $|T\rangle\langle T| \otimes I/2$ is magic, since discarding the second qubit, a stabilizer operation, leaves the T state. But the norm multiplies over tensor products, and $1.207 \times 0.5 < 1$:

In [ ]:
# T state on the first qubit, maximally mixed second qubit
t_and_mixed = qx.DensityMatrix.from_matrix(jnp.kron(t_matrix, mixed_matrix), (2, 2))
print(qx.stabilizer_norm(t_and_mixed))

So the stabilizer norm is not a magic monotone: discarding a qubit, a stabilizer operation, raised it from 0.60 to 1.21. Instead, it is a witness. A value above 1 proves magic, but the value itself is not a measure of how much.

## Stabilizer Rényi Entropies

For a pure state the characteristic distribution $\Xi$ is a probability distribution, and magic spreads it out. Leone, Oliviero and Hamma measure that spread with a Rényi entropy (Eq. (1) of ref.<sup><a href="#ref-8">8</a></sup>):

$$M_\alpha(|\psi\rangle) = \frac{1}{1 - \alpha} \log \sum_{x,z} \Xi_{x,z}^\alpha - \log D.$$

A stabilizer state has $\Xi = 1/D$ on $D$ entries, so the $-\log D$ shift makes it zero. $M_\alpha$ is zero exactly on pure stabilizer states, unchanged by Clifford gates, and additive over tensor products<sup><a href="#ref-8">8</a></sup>.

In [ ]:
# Zero state
print("|0>      ", qx.stabilizer_renyi_entropy(qx.states.KET0))

# T state, log(4/3)
print("|T>      ", qx.stabilizer_renyi_entropy(t_state))

# Two T states: twice as much
print("|T>|T>   ", qx.stabilizer_renyi_entropy(t_state | t_state))

# A Clifford gate does not change it
h_t_state = qx.apply_unitary_to_state_vector(qx.gates.H, t_state)
print("H|T>     ", qx.stabilizer_renyi_entropy(h_t_state))

# In bits instead of nats
print("in bits  ", qx.stabilizer_renyi_entropy(t_state, base=2))

The order $\alpha$ matters. For qubits, $M_\alpha$ is a magic monotone, never increased by stabilizer operations, for every integer $\alpha \ge 2$ (Theorem 1 of ref.<sup><a href="#ref-9">9</a></sup>), and it is not for $\alpha < 2$<sup><a href="#ref-10">10</a></sup>. quax defaults to $\alpha = 2$.

In [ ]:
print("alpha = 0", qx.stabilizer_renyi_entropy(t_state, alpha=0))
print("alpha = 1", qx.stabilizer_renyi_entropy(t_state, alpha=1))
print("alpha = 2", qx.stabilizer_renyi_entropy(t_state, alpha=2))
print("alpha = 3", qx.stabilizer_renyi_entropy(t_state, alpha=3))

The same formula works for qudits<sup><a href="#ref-11">11</a></sup>. The qutrit **strange state** $(|1\rangle - |2\rangle)/\sqrt{2}$, which appears again below, has $M_2 = \log 2$:

In [ ]:
# Strange state, (|1> - |2>) / sqrt(2)
strange = qx.StateVector.from_matrix(jnp.array([0, 1, -1], dtype=complex) / jnp.sqrt(2), (3,))

print("strange", qx.stabilizer_renyi_entropy(strange))
print("log 2  ", jnp.log(2))

**Mixed states.** For a mixed state $\Xi$ no longer sums to 1. Leone et al. correct for the state's own mixedness (Eq. (7) of ref.<sup><a href="#ref-8">8</a></sup>):

$$\tilde M_2(\rho) = -\log \frac{\sum_{x,z} |\chi_\rho(x,z)|^4}{\sum_{x,z} |\chi_\rho(x,z)|^2},$$

which equals $M_2$ on pure states. It is zero on the maximally mixed state, but not on every state in STAB: an equal mixture of $|0\rangle$ and $|+\rangle$ gets $\log(4/3)$, the same as the T state. Read it as a measure for nearly pure states.

In [ ]:
# Pure T state: the same as M_2
print("|T><T|     ", qx.mixed_stabilizer_renyi_entropy(t_state))

# Maximally mixed state: zero
print("I/2        ", qx.mixed_stabilizer_renyi_entropy(maximally_mixed))

# Equal mixture of |0> and |+>: a stabilizer state, but not zero
zero_matrix = qx.promote_state_vector_to_density_matrix(qx.states.KET0).matrix
plus_matrix = qx.promote_state_vector_to_density_matrix(qx.states.KETPLUS).matrix
zero_plus = qx.DensityMatrix.from_matrix((zero_matrix + plus_matrix) / 2, (2,))
print("(|0>+|+>)/2", qx.mixed_stabilizer_renyi_entropy(zero_plus))

## The Discrete Wigner Function

For odd $d$, a qudit has a phase space: a $d \times d$ grid of points $u = (q, p)$, where the position $q$ is the computational-basis index and $p$ is its Fourier partner, the momentum. Each point carries a **phase-point operator**, a Weyl-shifted copy of the parity operator $A_0|q\rangle = |-q\rangle$:

$$A_u = W_u A_0 W_u^\dagger, \qquad W_\rho(u) = \frac{1}{D}\,\mathrm{Tr}[A_u\,\rho].$$

This is Gross's Wigner function<sup><a href="#ref-1">1</a></sup>, as used by Veitch et al. (Sec. II B of ref.<sup><a href="#ref-3">3</a></sup>). It is real and sums to 1, and summing over $p$ gives the computational-basis probabilities. But it can go negative, and negative entries signal magic: by Hudson's theorem a pure state has a non-negative Wigner function exactly when it is a stabilizer state<sup><a href="#ref-1">1</a></sup>.

`qx.wigner_function` returns the grid indexed `W[q, p]`; on several qudits, `W[q_0, ..., q_{n-1}, p_0, ..., p_{n-1}]`. For even dimensions there is no such Wigner function, and quax raises an error.

In [ ]:
# Qutrit zero state; rows are q, columns are p
qutrit_zero = qx.zero_state_vector(dims=(3,))
print(qx.wigner_function(qutrit_zero))

# Strange state
print(qx.wigner_function(strange))

$|0\rangle$ sits on the line $q = 0$ with weight $1/3$ at each momentum. The strange state has one negative entry, $-1/3$. Its sums over $p$ reproduce the probabilities $|\langle q|\psi\rangle|^2$, and all twelve qutrit stabilizer states are non-negative:

In [ ]:
# Summing over p ...
print(qx.wigner_function(strange).sum(axis=1))

# ... gives the computational-basis probabilities
print(jnp.abs(strange.matrix) ** 2)

# The smallest entry over all twelve qutrit stabilizer states
print(qx.wigner_function(qx.stabilizer_states((3,))).min().round(12))

## Sum Negativity and Mana

Veitch et al. turn negativity into two magic monotones (Definitions 10 and 12, Theorems 11 and 13 of ref.<sup><a href="#ref-3">3</a></sup>). The **sum negativity** adds up the negative entries,

$$\mathrm{sn}(\rho) = \sum_{u:\,W_\rho(u) < 0} |W_\rho(u)| = \frac{1}{2}\Big(\sum_u |W_\rho(u)| - 1\Big),$$

and the **mana** takes a logarithm,

$$\mathcal{M}(\rho) = \log \sum_u |W_\rho(u)| = \log\big(2\,\mathrm{sn}(\rho) + 1\big).$$

The logarithm makes mana additive over tensor products, which the sum negativity is not<sup><a href="#ref-3">3</a></sup>. The **Norrell state** $(-|0\rangle + 2|1\rangle - |2\rangle)/\sqrt{6}$ has two entries of $-1/6$ instead of one of $-1/3$, and the same mana as the strange state.

In [ ]:
# Strange state: one entry of -1/3
print("sum negativity, strange          ", qx.sum_negativity(strange))
print("mana, strange                    ", qx.mana(strange))

# Two strange states: mana doubles, sum negativity does not
print("sum negativity, strange x strange", qx.sum_negativity(strange | strange))
print("mana, strange x strange          ", qx.mana(strange | strange))

# Norrell state, (-|0> + 2|1> - |2>) / sqrt(6): the same mana
norrell = qx.StateVector.from_matrix(jnp.array([-1, 2, -1], dtype=complex) / jnp.sqrt(6), (3,))
print("mana, norrell                    ", qx.mana(norrell))
print("log(5/3)                         ", jnp.log(5 / 3))

Both are zero on all of STAB, mixtures included, because mixing non-negative Wigner functions keeps them non-negative. Both are also zero on any magic state whose Wigner function happens to be non-negative. Such states belong to the *bound magic states*, which cannot be distilled into pure magic states<sup><a href="#ref-3">3</a></sup>; negativity cannot see them.

## The Magic of Channels

Wang, Wilde and Su carry these ideas from states to channels<sup><a href="#ref-12">12</a></sup>. A channel $\mathcal{N}$ moves each phase-point operator to a mixture of phase-point operators, which defines a table of conditional quasi-probabilities (Definition 2 of ref.<sup><a href="#ref-12">12</a></sup>):

$$W_\mathcal{N}(v|u) = \frac{1}{D_\mathrm{out}}\,\mathrm{Tr}[A_v\,\mathcal{N}(A_u)].$$

Each column sums to 1, and the table carries Wigner functions through the channel, $W_{\mathcal{N}(\rho)}(v) = \sum_u W_\mathcal{N}(v|u)\,W_\rho(u)$ (Lemma 1 of ref.<sup><a href="#ref-12">12</a></sup>).

**What makes a channel magic.** For states the free set is STAB. For channels, Wang, Wilde and Su take as free every channel that keeps non-negative Wigner functions non-negative, even when it acts on part of a larger system. They call these *completely positive-Wigner-preserving* channels (Definition 1 of ref.<sup><a href="#ref-12">12</a></sup>). They include every stabilizer operation, and circuits built from them can be simulated efficiently on a classical computer. A channel is free exactly when its table has no negative entry, so that it is an ordinary table of probabilities (Theorem 2 of ref.<sup><a href="#ref-12">12</a></sup>). A **magic channel** is one whose table goes negative somewhere. As with mana for states, this free set is larger than the stabilizer operations, so some channels that are not stabilizer operations still count as free.

`qx.wigner_function` computes the table for a `Unitary`, `SuperOp`, `Choi`, `KrausMap` or `PauliLiouville`, indexed `[q_out, p_out, q_in, p_in]`. A Clifford gate only moves points around, so its table is a permutation. The identity leaves every point in place; the shift $X$ moves position $q$ to $q + 1$:

In [ ]:
# Identity gate: every point stays where it is
identity = qx.Unitary.from_matrix(jnp.eye(3, dtype=complex), ((3,), (3,)))
print(qx.wigner_function(identity).reshape(9, 9))

# Shift gate X: |q> -> |q + 1>
shift = qx.Unitary.from_matrix(
    jnp.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]], dtype=complex),
    ((3,), (3,)),
)
print(qx.wigner_function(shift).reshape(9, 9))

A qutrit version of the T gate<sup><a href="#ref-13">13</a></sup>, $\mathrm{diag}(1, e^{2\pi i/9}, e^{-2\pi i/9})$, is not Clifford, and its table has negative entries: it is a magic channel. Propagating the strange state's Wigner function through it matches the Wigner function of the output state:

In [ ]:
# Qutrit T gate, diag(1, e^{2 pi i/9}, e^{-2 pi i/9})
qutrit_t = qx.Unitary.from_matrix(
    jnp.diag(jnp.exp(2j * jnp.pi * jnp.array([0, 1, -1]) / 9)),
    ((3,), (3,)),
)
table = qx.wigner_function(qutrit_t)
print("smallest table entry", table.min())

# Push the strange state's Wigner function through the table
propagated = jnp.tensordot(table, qx.wigner_function(strange), axes=2)

# Compare with the Wigner function of the output state
output_state = qx.apply_unitary_to_state_vector(qutrit_t, strange)
print("propagation matches ", jnp.allclose(propagated, qx.wigner_function(output_state)))

The **channel mana** is the largest mana the table can produce from a single phase-space point (Definition 4 of ref.<sup><a href="#ref-12">12</a></sup>):

$$\mathcal{M}(\mathcal{N}) = \log \max_u \sum_v |W_\mathcal{N}(v|u)|.$$

It is zero for Clifford gates and positive for the qutrit T gate. A channel that ignores its input and prepares the strange state has exactly the strange state's mana, $\log(5/3)$ (Proposition 4 of ref.<sup><a href="#ref-12">12</a></sup>).

In [ ]:
# Qutrit Fourier gate, a Clifford gate
omega = jnp.exp(2j * jnp.pi / 3)
fourier = qx.Unitary.from_matrix(
    jnp.array([[1, 1, 1], [1, omega, omega**2], [1, omega**2, omega]]) / jnp.sqrt(3),
    ((3,), (3,)),
)
print("Fourier        ", qx.mana(fourier))

# Qutrit T gate
print("qutrit T gate  ", qx.mana(qutrit_t))

# A channel that discards its input and prepares the strange state;
# its Kraus operators are |strange><0|, |strange><1| and |strange><2|
strange_ket = strange.matrix
prepare_strange = qx.KrausMap.from_matrix(
    jnp.stack(
        [
            jnp.outer(strange_ket, jnp.array([1, 0, 0])),
            jnp.outer(strange_ket, jnp.array([0, 1, 0])),
            jnp.outer(strange_ket, jnp.array([0, 0, 1])),
        ]
    ),
    ((3,), (3,)),
)
print("prepare strange", qx.mana(prepare_strange))
print("log(5/3)       ", jnp.log(5 / 3))

## Choosing a Measure

Start from what you have and follow the branches:

- **A channel or gate?**
  - **Odd $d$:** `qx.mana(channel)` gives one number, zero on Clifford gates; `qx.wigner_function(channel)` shows the table behind it.
  - **Even $d$:** quax has no channel measure.
- **A state?**
  - **Odd $d$:** `qx.mana(state)` is a monotone, additive, and zero on all of STAB, but misses magic whose Wigner function is non-negative. `qx.sum_negativity` and `qx.wigner_function` show where the negativity is.
  - **Any $d$, pure:** `qx.stabilizer_renyi_entropy(state)` is zero exactly on stabilizer states, additive, and, for qubits, a monotone for integer $\alpha \ge 2$.
  - **Any $d$, mixed:**
    - `qx.stabilizer_norm(state)` above 1 proves magic. It is exact for one qubit and can miss magic beyond that.
    - `qx.mixed_stabilizer_renyi_entropy(state)` suits nearly pure states; it is not zero on every stabilizer mixture.
- **A zero-magic reference?** `qx.stabilizer_states(dims)` lists every pure stabilizer state for prime $d$.

| Function | Input | $d$ | Zero on | Monotone | Sources |
|---|---|---|---|---|---|
| `characteristic_function`, `characteristic_distribution` | state, observable | any | | | <sup><a href="#ref-1">1</a>,<a href="#ref-2">2</a></sup> |
| `stabilizer_states` | dimensions | prime | | | <sup><a href="#ref-1">1</a></sup> |
| `stabilizer_norm` | state | any | equals 1 on pure stabilizer states, at most 1 on STAB | no (witness) | <sup><a href="#ref-6">6</a>,<a href="#ref-7">7</a></sup> |
| `stabilizer_renyi_entropy` | pure state | any | exactly the pure stabilizer states | for qubits, integer $\alpha \ge 2$ | <sup><a href="#ref-8">8</a>,<a href="#ref-9">9</a>,<a href="#ref-10">10</a>,<a href="#ref-11">11</a></sup> |
| `mixed_stabilizer_renyi_entropy` | state | any | pure stabilizer states, $I/D$, stabilizer-code projectors | no | <sup><a href="#ref-8">8</a></sup> |
| `wigner_function` | state, observable, channel | odd | | | <sup><a href="#ref-1">1</a>,<a href="#ref-3">3</a>,<a href="#ref-12">12</a></sup> |
| `sum_negativity` | state | odd | all of STAB, and magic states with non-negative Wigner functions | yes | <sup><a href="#ref-3">3</a></sup> |
| `mana` | state | odd | all of STAB, and magic states with non-negative Wigner functions | yes | <sup><a href="#ref-3">3</a></sup> |
| `mana` | channel | odd | channels that completely preserve non-negative Wigner functions, Clifford gates included | yes | <sup><a href="#ref-12">12</a></sup> |

## References

1. <a id="ref-1"></a>D. Gross, "Hudson's theorem for finite-dimensional quantum systems," *J. Math. Phys.* **47**, 122107 (2006). [arXiv:quant-ph/0602001](https://arxiv.org/abs/quant-ph/0602001)
2. <a id="ref-2"></a>D. Gross, S. Nezami, and M. Walter, "Schur–Weyl duality for the Clifford group with applications: property testing, a robust Hudson theorem, and de Finetti representations," *Commun. Math. Phys.* **385**, 1325 (2021). [arXiv:1712.08628](https://arxiv.org/abs/1712.08628)
3. <a id="ref-3"></a>V. Veitch, S. A. H. Mousavian, D. Gottesman, and J. Emerson, "The resource theory of stabilizer quantum computation," *New J. Phys.* **16**, 013009 (2014). [arXiv:1307.7171](https://arxiv.org/abs/1307.7171)
4. <a id="ref-4"></a>D. Gottesman, "The Heisenberg representation of quantum computers," in *Proceedings of the XXII International Colloquium on Group Theoretical Methods in Physics*, pp. 32–43 (1999). [arXiv:quant-ph/9807006](https://arxiv.org/abs/quant-ph/9807006)
5. <a id="ref-5"></a>S. Bravyi and A. Kitaev, "Universal quantum computation with ideal Clifford gates and noisy ancillas," *Phys. Rev. A* **71**, 022316 (2005). [arXiv:quant-ph/0403025](https://arxiv.org/abs/quant-ph/0403025)
6. <a id="ref-6"></a>E. T. Campbell, "Catalysis and activation of magic states in fault-tolerant architectures," *Phys. Rev. A* **83**, 032317 (2011). [arXiv:1010.0104](https://arxiv.org/abs/1010.0104)
7. <a id="ref-7"></a>M. Howard and E. Campbell, "Application of a resource theory for magic states to fault-tolerant quantum computing," *Phys. Rev. Lett.* **118**, 090501 (2017). [arXiv:1609.07488](https://arxiv.org/abs/1609.07488)
8. <a id="ref-8"></a>L. Leone, S. F. E. Oliviero, and A. Hamma, "Stabilizer Rényi entropy," *Phys. Rev. Lett.* **128**, 050402 (2022). [arXiv:2106.12587](https://arxiv.org/abs/2106.12587)
9. <a id="ref-9"></a>L. Leone and L. Bittel, "Stabilizer entropies are monotones for magic-state resource theory," *Phys. Rev. A* **110**, L040403 (2024). [arXiv:2404.11652](https://arxiv.org/abs/2404.11652)
10. <a id="ref-10"></a>T. Haug and L. Piroli, "Stabilizer entropies and nonstabilizerness monotones," *Quantum* **7**, 1092 (2023). [arXiv:2303.10152](https://arxiv.org/abs/2303.10152)
11. <a id="ref-11"></a>I. Klich, "SWAP and transpose by displacements, stabilizer Rényi entropies for continuous variables and qudits and other applications," arXiv preprint (2024). [arXiv:2408.15161](https://arxiv.org/abs/2408.15161)
12. <a id="ref-12"></a>X. Wang, M. M. Wilde, and Y. Su, "Quantifying the magic of quantum channels," *New J. Phys.* **21**, 103002 (2019). [arXiv:1903.04483](https://arxiv.org/abs/1903.04483)
13. <a id="ref-13"></a>M. Howard and J. Vala, "Qudit versions of the qubit π/8 gate," *Phys. Rev. A* **86**, 022316 (2012). [arXiv:1206.1598](https://arxiv.org/abs/1206.1598)